## Section 1 — Setup and Feature String Construction

We load a working sample from the warehouse and build the feature string for each article. The sample is drawn as a stratified set — equal numbers from each segment — so neither system is evaluated on a skewed distribution.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from pathlib import Path
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# --- Locate project root and load credentials ---
# We walk up from the notebook directory until we find environment.yml.
# This makes the notebook runnable from any working directory.
def find_project_root():
    current = Path('.').resolve()
    while current != current.parent:
        if (current / 'environment.yml').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_USER = os.getenv('DB_USER', 'root')
DB_PASS = os.getenv('DB_PASSWORD')
DB_NAME = os.getenv('DB_NAME', 'news_pulse')

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    pool_pre_ping=True
)

print('Environment loaded.')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# --- Load stratified sample from warehouse ---
#
# Why stratified?
# A random sample from fact_articles would reflect the actual segment
# distribution, which is skewed (Politics dominates after the SEGMENT_MAP
# fix). A stratified sample gives each segment equal representation so
# evaluation metrics are not dominated by one segment.
#
# Why 500 per segment?
# 3,500 total articles (7 segments x 500) is large enough for meaningful
# evaluation while keeping similarity matrix computation fast.
# TF-IDF on 3,500 docs is near-instant. Sentence transformers on 3,500
# docs takes ~2 minutes on CPU.

ARTICLES_PER_SEGMENT = 500

query = text("""
    SELECT
        fa.article_id,
        fa.url,
        fa.seendate,
        ds_src.source_name,
        ds_seg.segment_name,
        GROUP_CONCAT(
            DISTINCT de.entity_name
            ORDER BY de.entity_name
            SEPARATOR ' '
        ) AS entities
    FROM fact_articles fa
    JOIN dim_source ds_src   ON fa.source_id  = ds_src.source_id
    JOIN dim_segment ds_seg  ON fa.segment_id = ds_seg.segment_id
    LEFT JOIN fact_entity_mentions fem ON fa.article_id = fem.article_id
    LEFT JOIN dim_entity de            ON fem.entity_id = de.entity_id
    WHERE ds_seg.segment_name != 'General'
    GROUP BY fa.article_id, fa.url, fa.seendate, ds_src.source_name, ds_seg.segment_name
    LIMIT 50000
""")

print('Fetching articles from warehouse...')
with engine.connect() as conn:
    df_all = pd.read_sql(query, conn)

print(f'Total fetched: {len(df_all):,} articles')
print(f'Segment distribution:\n{df_all["segment_name"].value_counts().to_string()}')

In [ ]:
# --- Stratify: take up to ARTICLES_PER_SEGMENT from each segment ---

frames = []
for segment, group in df_all.groupby('segment_name'):
    sample = group.sample(
        n=min(ARTICLES_PER_SEGMENT, len(group)),
        random_state=42
    )
    frames.append(sample)

df = pd.concat(frames, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Working sample: {len(df):,} articles')
print(f'\nStratified distribution:')
print(df['segment_name'].value_counts().to_string())

In [ ]:
# --- Build synthetic feature strings ---
#
# Each article becomes one string combining:
#   - source_name: the outlet, which carries topical signal
#     (e.g. espn.com is almost always Sports)
#   - segment_name: the segment, with spaces replaced by underscores
#     so TF-IDF treats it as one token not two
#   - themes: raw GDELT theme tags, semicolon-separated
#     (e.g. ECON_POVERTY;LEADER;WB_855_LABOR)
#   - entities: named persons and organisations from dim_entity
#
# Why replace spaces with underscores in multi-word terms?
# TF-IDF splits on whitespace by default. "Business and Markets" becomes
# three tokens: Business, and, Markets. "Business_and_Markets" stays as
# one meaningful token. Same logic applies to entity names.

def build_feature_string(row):
    parts = []

    if pd.notna(row['source_name']) and str(row['source_name']).strip():
        source = re.sub(r'[^\w]', '_', str(row['source_name']).strip().lower())
        parts.append(source)

    if pd.notna(row['segment_name']):
        parts.append(row['segment_name'].replace(' ', '_'))

    if pd.notna(row['entities']) and str(row['entities']).strip():
        names = str(row['entities']).split(' ')
        for name in names:
            clean = name.strip()
            if clean:
                parts.append(clean.replace(' ', '_'))

    return ' '.join(parts) if parts else 'unknown'


df['feature_string'] = df.apply(build_feature_string, axis=1)

print('Feature strings built.')
print(f'\nSample feature strings:')
for _, row in df.sample(3, random_state=1).iterrows():
    print(f'  [{row["segment_name"]}] {row["feature_string"][:120]}...')
    print()

# Sanity check: any empty feature strings?
empty = (df['feature_string'] == 'unknown').sum()
print(f'Empty feature strings: {empty} / {len(df)}')

## Section 2 — TF-IDF Baseline

TF-IDF (Term Frequency–Inverse Document Frequency) represents each article as a vector of word weights. Words that appear often in one article but rarely across all articles get high weight — they are distinctive. Common words like "the" or "and" get near-zero weight.

Cosine similarity between two vectors measures how similar their word distributions are, regardless of document length. Two articles sharing many distinctive terms score high; two articles sharing only common words score low.

The key limitation here: TF-IDF is vocabulary-matching. It cannot know that "ECON_POVERTY" and "WB_855_LABOR" are related topics.

In [ ]:
# --- Build TF-IDF matrix ---
#
# min_df=2: ignore tokens that appear in only one article.
#   Single-occurrence tokens add noise without helping similarity.
# max_df=0.95: ignore tokens that appear in 95%+ of articles.
#   Near-universal tokens carry no discriminating signal.
# ngram_range=(1,2): include both single tokens and adjacent pairs.
#   This lets the model capture "Federal Reserve" as a unit, not just
#   "Federal" and "Reserve" separately.
# sublinear_tf=True: applies log scaling to term frequency.
#   Prevents articles with many repeated tokens from dominating similarity.

print('Building TF-IDF matrix...')

tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df['feature_string'])

print(f'Matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,} tokens')
print(f'Matrix density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f} (sparse)')

In [ ]:
# --- TF-IDF recommendation function ---
#
# Given an article index, compute cosine similarity against all other
# articles and return the top N most similar.
#
# We exclude the query article itself from results (it would always score 1.0).

def recommend_tfidf(article_idx, n=5):
    """
    Returns the top-N most similar articles to the article at article_idx,
    using TF-IDF cosine similarity.
    """
    query_vec = tfidf_matrix[article_idx]
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    scores[article_idx] = 0  # exclude self

    top_indices = scores.argsort()[::-1][:n]

    results = df.iloc[top_indices][['article_id', 'source_name', 'segment_name', 'url']].copy()
    results['similarity_score'] = scores[top_indices].round(4)
    results['rank'] = range(1, n + 1)
    return results


# Spot-check on one article from each segment
print('TF-IDF spot checks — top 3 recommendations per query article\n')
print('=' * 70)

for segment in df['segment_name'].unique():
    sample_row = df[df['segment_name'] == segment].iloc[0]
    idx = sample_row.name
    recs = recommend_tfidf(idx, n=3)

    print(f'\nQuery [{segment}]: {sample_row["source_name"]}')
    print(f'Feature string preview: {sample_row["feature_string"][:100]}...')
    print('Recommendations:')
    for _, rec in recs.iterrows():
        match = '✓ same segment' if rec['segment_name'] == segment else '✗ different segment'
        print(f'  {rec["rank"]}. [{rec["segment_name"]}] {rec["source_name"]} — score: {rec["similarity_score"]} {match}')

## Section 3 — Sentence Transformer Recommender

Sentence transformers encode each article as a dense 384-dimensional vector trained to capture semantic meaning. Similar concepts map to nearby points in that 384-dimensional space, regardless of whether they share exact vocabulary.

We use `all-MiniLM-L6-v2` — a small, fast model that runs comfortably on CPU and produces strong results for short-to-medium text. It was trained on over 1 billion sentence pairs and benchmarks well for semantic similarity tasks.

The tradeoff vs TF-IDF: encoding 3,500 articles takes ~2 minutes on CPU versus milliseconds for TF-IDF. Once the embeddings are built, lookup is equally fast.

In [ ]:
# --- Build sentence transformer embeddings ---
#
# encode() processes all feature strings in one batch.
# show_progress_bar=True displays a tqdm progress bar — useful here
# because 3,500 articles takes ~2 minutes on CPU.
#
# convert_to_numpy=True returns a standard numpy array rather than
# a PyTorch tensor, which is what cosine_similarity expects.

print('Loading sentence transformer model (all-MiniLM-L6-v2)...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded.')

print('\nEncoding feature strings — this takes ~2 minutes on CPU...')
embeddings = st_model.encode(
    df['feature_string'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Each article is a {embeddings.shape[1]}-dimensional vector.')

In [ ]:
# --- Sentence transformer recommendation function ---

def recommend_st(article_idx, n=5):
    """
    Returns the top-N most similar articles to the article at article_idx,
    using sentence transformer cosine similarity.
    """
    query_vec = embeddings[article_idx].reshape(1, -1)
    scores = cosine_similarity(query_vec, embeddings).flatten()
    scores[article_idx] = 0  # exclude self

    top_indices = scores.argsort()[::-1][:n]

    results = df.iloc[top_indices][['article_id', 'source_name', 'segment_name', 'url']].copy()
    results['similarity_score'] = scores[top_indices].round(4)
    results['rank'] = range(1, n + 1)
    return results


# Spot-check on the same articles used for TF-IDF
print('Sentence transformer spot checks — top 3 recommendations per query article\n')
print('=' * 70)

for segment in df['segment_name'].unique():
    sample_row = df[df['segment_name'] == segment].iloc[0]
    idx = sample_row.name
    recs = recommend_st(idx, n=3)

    print(f'\nQuery [{segment}]: {sample_row["source_name"]}')
    print(f'Feature string preview: {sample_row["feature_string"][:100]}...')
    print('Recommendations:')
    for _, rec in recs.iterrows():
        match = '✓ same segment' if rec['segment_name'] == segment else '✗ different segment'
        print(f'  {rec["rank"]}. [{rec["segment_name"]}] {rec["source_name"]} — score: {rec["similarity_score"]} {match}')

## Section 4 — Quantitative Comparison

We compare both systems on three proxy metrics:

**Intra-segment precision (P@5):** For each article, what fraction of the top 5 recommendations are from the same segment? Higher is better — a Sports article should recommend other Sports articles.

**Inter-segment contamination rate:** What fraction of recommendations come from a different segment? The inverse of precision. Lower is better.

**Mean similarity score:** The average cosine similarity of the top 5 recommendations. Higher scores mean the system is finding more confident matches. Note: TF-IDF and sentence transformer scores are not directly comparable in absolute value — what matters is the *relative* difference between systems and the spread of scores.

These are honest proxies. Segment membership is a reasonable stand-in for topical relevance, but it is not a ground truth label. A Sports article recommending another Sports article is likely good; it is not guaranteed to be.

In [ ]:
# --- Evaluate both systems across all articles ---
#
# For each article in the sample, we get the top 5 recommendations
# from both systems and record:
#   - how many are in the same segment (precision)
#   - the mean similarity score of the top 5
#
# We evaluate on a subset of 700 articles (100 per segment) rather than
# the full 3,500 to keep runtime reasonable. Results are representative
# because the sample is stratified.

EVAL_PER_SEGMENT = 100
TOP_N = 5

print(f'Evaluating both systems on {EVAL_PER_SEGMENT} articles per segment (top {TOP_N} recs each)...')

eval_frames = []
for segment, group in df.groupby('segment_name'):
    eval_frames.append(group.sample(n=min(EVAL_PER_SEGMENT, len(group)), random_state=99))

eval_df = pd.concat(eval_frames, ignore_index=False)  # keep original index for matrix lookup
print(f'Evaluation set: {len(eval_df)} articles')

tfidf_results = []
st_results    = []

for idx in eval_df.index:
    query_segment = df.loc[idx, 'segment_name']

    # TF-IDF
    recs_tf = recommend_tfidf(idx, n=TOP_N)
    precision_tf = (recs_tf['segment_name'] == query_segment).mean()
    mean_sim_tf  = recs_tf['similarity_score'].mean()
    tfidf_results.append({
        'segment': query_segment,
        'precision': precision_tf,
        'mean_similarity': mean_sim_tf
    })

    # Sentence transformer
    recs_st = recommend_st(idx, n=TOP_N)
    precision_st = (recs_st['segment_name'] == query_segment).mean()
    mean_sim_st  = recs_st['similarity_score'].mean()
    st_results.append({
        'segment': query_segment,
        'precision': precision_st,
        'mean_similarity': mean_sim_st
    })

tfidf_eval = pd.DataFrame(tfidf_results)
st_eval    = pd.DataFrame(st_results)

# Overall summary
print('\n--- Overall Results ---')
print(f'TF-IDF    — P@{TOP_N}: {tfidf_eval["precision"].mean():.3f} | Mean similarity: {tfidf_eval["mean_similarity"].mean():.4f}')
print(f'Sen. Trans — P@{TOP_N}: {st_eval["precision"].mean():.3f} | Mean similarity: {st_eval["mean_similarity"].mean():.4f}')

# Per-segment breakdown
print(f'\n--- P@{TOP_N} by Segment ---')
seg_compare = pd.DataFrame({
    'TF-IDF': tfidf_eval.groupby('segment')['precision'].mean().round(3),
    'SentenceTransformer': st_eval.groupby('segment')['precision'].mean().round(3)
})
seg_compare['ST_advantage'] = (seg_compare['SentenceTransformer'] - seg_compare['TF-IDF']).round(3)
print(seg_compare.to_string())

In [ ]:
# --- Visualise the comparison ---
#
# Three charts:
#   1. P@5 by segment for both systems — side-by-side bars
#   2. Similarity score distributions — boxplot showing spread
#   3. ST advantage heatmap — which segments benefit most

SEGMENT_ORDER = sorted(df['segment_name'].unique())

fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Colour palette
C_TFIDF = '#4C72B0'
C_ST    = '#DD8452'
C_ADV   = '#55A868'

# --- Chart 1: P@5 by segment, side-by-side ---
x = np.arange(len(SEGMENT_ORDER))
w = 0.35

tfidf_prec = [tfidf_eval[tfidf_eval['segment'] == s]['precision'].mean() for s in SEGMENT_ORDER]
st_prec    = [st_eval[st_eval['segment'] == s]['precision'].mean() for s in SEGMENT_ORDER]

bars1 = ax1.bar(x - w/2, tfidf_prec, width=w, label='TF-IDF', color=C_TFIDF, alpha=0.85)
bars2 = ax1.bar(x + w/2, st_prec,    width=w, label='Sentence Transformer', color=C_ST, alpha=0.85)

ax1.set_xticks(x)
ax1.set_xticklabels([s.replace(' and ', '\n& ') for s in SEGMENT_ORDER], fontsize=10)
ax1.set_ylabel('Precision @ 5', fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.axhline(0.2, color='grey', linestyle='--', linewidth=0.8, alpha=0.5,
            label='Random baseline (1 in 7 segments)')
ax1.legend(fontsize=10)
ax1.set_title('Intra-Segment Precision @ 5 by Segment', fontsize=13, pad=10)

for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
             ha='center', va='bottom', fontsize=8, color='#333333')
for bar in bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
             ha='center', va='bottom', fontsize=8, color='#333333')

# --- Chart 2: Similarity score distributions ---
sim_data = [
    tfidf_eval['mean_similarity'].values,
    st_eval['mean_similarity'].values
]
bp = ax2.boxplot(sim_data, patch_artist=True, widths=0.5,
                 medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor(C_TFIDF)
bp['boxes'][0].set_alpha(0.75)
bp['boxes'][1].set_facecolor(C_ST)
bp['boxes'][1].set_alpha(0.75)

ax2.set_xticklabels(['TF-IDF', 'Sentence\nTransformer'], fontsize=10)
ax2.set_ylabel('Mean Similarity Score (top 5)', fontsize=10)
ax2.set_title('Similarity Score Distribution', fontsize=12, pad=8)
ax2.text(0.5, -0.18,
         'Note: TF-IDF and sentence transformer scores are not comparable\nin absolute value — compare spread, not level.',
         transform=ax2.transAxes, ha='center', fontsize=8, color='#555555')

# --- Chart 3: ST advantage by segment ---
advantages = seg_compare['ST_advantage'].reindex(SEGMENT_ORDER)
colors = [C_ADV if v >= 0 else '#CC4125' for v in advantages]

ax3.barh(SEGMENT_ORDER, advantages.values, color=colors, alpha=0.85)
ax3.axvline(0, color='black', linewidth=0.8)
ax3.set_xlabel('ST precision minus TF-IDF precision', fontsize=10)
ax3.set_title('Sentence Transformer Advantage\nby Segment', fontsize=12, pad=8)

for i, (seg, val) in enumerate(zip(SEGMENT_ORDER, advantages.values)):
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    ax3.text(val + offset, i, f'{val:+.3f}', va='center', ha=ha,
             fontsize=9, color='#333333')

fig.suptitle('Recommendation Engine — TF-IDF vs Sentence Transformer',
             fontsize=15, y=1.01, fontweight='bold')

plt.savefig(PROJECT_ROOT / 'reports' / '09_recommender_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/09_recommender_comparison.png')

In [ ]:
# --- Cross-segment confusion: where does each system fail? ---
#
# For each query segment, what segment does the system most often
# recommend instead when it gets it wrong?
# This reveals whether failures are random noise or systematic
# (e.g. Politics articles being confused with Business articles).

def build_confusion(eval_df_input, recommend_fn, top_n=5):
    rows = []
    for idx in eval_df_input.index:
        query_seg = df.loc[idx, 'segment_name']
        recs = recommend_fn(idx, n=top_n)
        for _, rec in recs.iterrows():
            rows.append({'query': query_seg, 'recommended': rec['segment_name']})
    pair_df = pd.DataFrame(rows)
    confusion = pair_df.groupby(['query', 'recommended']).size().unstack(fill_value=0)
    # Normalise by row so each row sums to 1
    confusion = confusion.div(confusion.sum(axis=1), axis=0).round(3)
    return confusion

print('Building confusion matrices (this may take ~1 minute)...')
confusion_tfidf = build_confusion(eval_df, recommend_tfidf)
confusion_st    = build_confusion(eval_df, recommend_st)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, matrix, title in [
    (axes[0], confusion_tfidf, 'TF-IDF — Recommendation Distribution'),
    (axes[1], confusion_st,    'Sentence Transformer — Recommendation Distribution')
]:
    # Reindex to consistent segment order
    matrix = matrix.reindex(index=SEGMENT_ORDER, columns=SEGMENT_ORDER, fill_value=0)
    sns.heatmap(
        matrix,
        ax=ax,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        linewidths=0.5,
        vmin=0,
        vmax=1,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel('Recommended segment', fontsize=10)
    ax.set_ylabel('Query segment', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

fig.suptitle('Segment Confusion: Where Each System Gets It Wrong',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / '10_segment_confusion.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/10_segment_confusion.png')

## Section 5 — Summary Findings

Record the actual results here after running the notebook. Replace the placeholder text with the real numbers.

In [ ]:
# --- Summary Findings ---
#
# Written after running the full evaluation. These findings feed
# directly into recommend.py design decisions below.

print("""
RECOMMENDATION ENGINE — FINDINGS SUMMARY
=========================================

Input signal
------------
Feature strings were built from source name, segment label, and named
entities (persons and organisations from dim_entity). GDELT theme tags
are not stored in fact_articles, so they could not be included. Article
text is not stored at all — the pipeline ingests metadata only. Both
systems therefore operate on structured metadata, not prose.

The Quotations field was investigated during notebook planning. Fill rate
was 15% across a live GKG sample, with complex pipe-delimited formatting
(offset|length||text). Including it would have added parsing complexity
for inconsistent coverage, so it was excluded. Option 3 — synthetic
feature strings — was chosen as the uniform approach across all 11.9M
historical rows and all future ingests.

System 1: TF-IDF baseline
--------------------------
Overall P@5: 0.657
Best segment:  Crime and Justice (0.754)
Worst segment: Politics and Government (0.572)

TF-IDF performs reasonably well for a metadata-only system. Crime and
Justice scores highest because crime articles carry highly specific named
entities — courts, suspects, locations — that are rare enough to be
discriminating tokens. Politics and Government scores lowest because
political articles share many common entities (Trump, Biden, Congress)
across different story angles, so shared vocabulary does not reliably
signal topical similarity.

The similarity score distribution is wide and erratic. The median sits
around 0.34 with occasional outliers above 0.80. This means TF-IDF
produces inconsistent confidence — sometimes very sure, often uncertain.

System 2: Sentence transformer (all-MiniLM-L6-v2)
---------------------------------------------------
Overall P@5: 0.878
Best segment:  Crime and Justice (0.988)
Worst segment: Technology (0.824)

The sentence transformer outperforms TF-IDF in every segment without
exception. The overall advantage is +0.221 percentage points. The
similarity score distribution is tighter (median ~0.67, narrower IQR),
meaning the system produces consistently confident matches rather than
erratic ones.

Politics and Government shows the largest gain (+0.290). This is the
segment where TF-IDF suffers most — shared political vocabulary confuses
it — and where semantic understanding of meaning over vocabulary pays off
most. Science and Health follows at +0.262 for similar reasons.

Technology shows the smallest gain (+0.120). This segment had only 884
articles in the full 50,000 before stratification, the smallest coverage
by far. Thin data limits both systems.

Confusion matrix findings
--------------------------
TF-IDF bleeds most from Politics into Business and Markets (0.11) and
Science and Health (0.10). These are genuinely overlapping domains in
GDELT coverage — economic policy and public health both attract political
entities — so the confusion is structural, not random noise.

Entertainment and Culture is TF-IDF's messiest row, bleeding into Crime
(0.09), Science (0.11), and Business (0.07). The sentence transformer
tightens this substantially.

The sentence transformer confusion matrix is close to diagonal across all
segments. The off-diagonal values are mostly 0.00 to 0.04. The remaining
cross-contamination between Business and Politics (0.06) is the only
notable bleed, again reflecting genuine domain overlap.

Data quality note
------------------
Spot checks revealed the same URL appearing multiple times in top-N
results for some articles (e.g. hospitalitynet.org appearing 3 times for
a Technology query). This indicates near-duplicate records in the warehouse
from the same source. recommend.py deduplicates by URL before returning
results to prevent this appearing in production output.

Design decision
---------------
The sentence transformer is selected as the production system for
recommend.py. TF-IDF is retained as a documented baseline. The gap is
large enough (+0.221 overall) and consistent enough (all six segments)
that using TF-IDF in production would be a knowable downgrade.

The evaluation metric — intra-segment precision — is a proxy, not ground
truth. Segment membership is a reasonable stand-in for topical relevance
but is not a human relevance label. Results should be interpreted with
that constraint in mind.
""")